# 2. Hexagons and Households

The map divides Montgomery County into a grid of hexagons a quarter-mile apart and gives each hexagon its share of the county's homes. This notebook rebuilds that grid and those shares from the Census files alone, then checks them against the travel-time file behind the map:

1. **The 422 Census block groups**: how many households live in them and how many have no car.
2. **The 8,568 hexagons**: laid out on a regular grid, each one belonging to the block group its center falls in.
3. **The households in each hexagon**: apportioned using 2020 Census block housing counts so every block group keeps its exact household total.
4. **The 2,933 lived-in hexagons**: the ones dense enough to count toward a neighborhood.

**How to read this notebook:** Each grey box runs a small piece of Python and prints what it found right below it. A box that starts with `assert` is a check: if any number disagrees with what the site publishes, the notebook stops with an error. Each check is written out in words just above it.

In [1]:
import json
import math
import statistics
from collections import Counter, defaultdict
from pathlib import Path

snapshots_folder = Path('..') / 'snapshots'


def count_of(items):
    return len(items)

## 1. The 422 Census block groups

The U.S. Census Bureau's American Community Survey (ACS) reports total occupied households and carless households for each of the county's 422 block groups (421 of which have residents).

In [2]:
block_group_layer = json.loads((snapshots_folder / 'Quantile_Vehicle@2026-10-03.geojson').read_text())

block_groups = [
    {
        'geoid': feature['properties']['GEOID'],
        'households': feature['properties']['Total_Occupied_Housing_Units'] or 0,
        'carless_households': feature['properties']['No_Vehicle'],
        'carless_percent': feature['properties']['PercentNoVehicle'],
        'interior_point': {'lat': float(feature['properties']['INTPTLAT']), 'lon': float(feature['properties']['INTPTLON'])},
        'rings': feature['geometry']['coordinates'],
    }
    for feature in block_group_layer['features']
]
lived_in_block_groups = [block_group for block_group in block_groups if block_group['households'] > 0]

print(f'{count_of(block_groups)} block groups, {count_of(lived_in_block_groups)} with households')

422 block groups, 421 with households


In [3]:
county_households = sum(block_group['households'] for block_group in lived_in_block_groups)
county_carless_households = sum(block_group['carless_households'] for block_group in lived_in_block_groups)

print(f'{county_households:,} households, {county_carless_households:,} without a car ({county_carless_households / county_households:.1%})')

231,319 households, 19,669 without a car (8.5%)


We flag a block group as **high vehicle need** when more than **15%** of its households do not have a car, nearly double the countywide rate of 8.5%.

In [4]:
carless_line_percent = 15

high_vehicle_need_block_groups = [block_group for block_group in lived_in_block_groups if block_group['carless_percent'] > carless_line_percent]
high_vehicle_need_households = sum(block_group['households'] for block_group in high_vehicle_need_block_groups)
high_vehicle_need_carless_households = sum(block_group['carless_households'] for block_group in high_vehicle_need_block_groups)

print(f'{count_of(high_vehicle_need_block_groups)} block groups over {carless_line_percent}% carless')
print(f'home to {high_vehicle_need_households:,} households, {high_vehicle_need_carless_households:,} of them without a car')

84 block groups over 15% carless
home to 40,328 households, 11,137 of them without a car


**Check:** 422 block groups; 231,319 households, 19,669 of them without a car; 84 block groups over 15% carless, home to 11,137 carless households.

In [5]:
assert count_of(block_groups) == 422
assert county_households == 231_319
assert county_carless_households == 19_669
assert count_of(high_vehicle_need_block_groups) == 84
assert high_vehicle_need_carless_households == 11_137

## 2. The 8,568 hexagons

Starting at the southwest corner of the county, we lay out a honeycomb grid of hexagons spaced a quarter-mile apart, with every other row shifted half a step east so every hexagon has six equidistant neighbors.

In [6]:
earth_radius_mi = 3958.8
radians_per_degree = math.pi / 180

boundary_points = []
for block_group in block_groups:
    for ring in block_group['rings']:
        boundary_points.extend(ring)
south_lat = min(lat for lon, lat in boundary_points)
north_lat = max(lat for lon, lat in boundary_points)
west_lon = min(lon for lon, lat in boundary_points)
east_lon = max(lon for lon, lat in boundary_points)

miles_per_degree_lat = earth_radius_mi * radians_per_degree
miles_per_degree_lon = earth_radius_mi * radians_per_degree * math.cos((south_lat + north_lat) / 2 * radians_per_degree)

north_to_south_mi = (north_lat - south_lat) * miles_per_degree_lat
east_to_west_mi = (east_lon - west_lon) * miles_per_degree_lon
print(f'county box: {north_to_south_mi:.2f} mi north to south, {east_to_west_mi:.2f} mi east to west')

county box: 23.82 mi north to south, 23.09 mi east to west


In [7]:
hexagon_spacing_mi = 0.25
row_spacing_mi = hexagon_spacing_mi * math.sqrt(3) / 2
county_height_mi = (north_lat - south_lat) * miles_per_degree_lat
county_width_mi = (east_lon - west_lon) * miles_per_degree_lon


def hexagon_center(row, column):
    odd_row_shift_mi = hexagon_spacing_mi / 2 if row % 2 == 1 else 0
    return {
        'lat': south_lat + row * row_spacing_mi / miles_per_degree_lat,
        'lon': west_lon + (column * hexagon_spacing_mi + odd_row_shift_mi - hexagon_spacing_mi / 2) / miles_per_degree_lon,
    }


grid_positions = [
    {'id': f'r{row}c{column}', 'row': row, 'column': column, **hexagon_center(row, column)}
    for row in range(int((county_height_mi + row_spacing_mi) / row_spacing_mi) + 1)
    for column in range(int((county_width_mi + hexagon_spacing_mi) / hexagon_spacing_mi) + 1)
]

print(f'{count_of(grid_positions):,} grid positions covering the county box')

10,528 grid positions covering the county box


Each hexagon belongs to the block group that contains its center. To test whether a center point falls inside a boundary, we trace a line due east and count how many times it crosses the boundary: an odd number of crossings means the point is inside. A point inside a hole in the boundary, such as a lake, does not count.

In [8]:
def boundary_edges(ring):
    return zip(ring, ring[-1:] + ring[:-1])


def ring_contains(point, ring):
    crossings = 0
    for (corner_lon, corner_lat), (previous_lon, previous_lat) in boundary_edges(ring):
        edge_spans_point_lat = (corner_lat > point['lat']) != (previous_lat > point['lat'])
        if not edge_spans_point_lat:
            continue
        edge_lon_at_point_lat = corner_lon + (point['lat'] - corner_lat) / (previous_lat - corner_lat) * (previous_lon - corner_lon)
        if point['lon'] < edge_lon_at_point_lat:
            crossings += 1
    return crossings % 2 == 1


def block_group_contains(block_group, point):
    outer_ring = block_group['rings'][0]
    holes = block_group['rings'][1:]
    return ring_contains(point, outer_ring) and not any(ring_contains(point, hole) for hole in holes)

In [9]:
for block_group in block_groups:
    ring_points = [point for ring in block_group['rings'] for point in ring]
    ring_lons = [lon for lon, lat in ring_points]
    ring_lats = [lat for lon, lat in ring_points]
    block_group['box'] = (min(ring_lons), max(ring_lons), min(ring_lats), max(ring_lats))


def block_group_at(point):
    for block_group in block_groups:
        west, east, south, north = block_group['box']
        if west <= point['lon'] <= east and south <= point['lat'] <= north and block_group_contains(block_group, point):
            return block_group['geoid']
    return None


hexagons_by_id = {}
for position in grid_positions:
    block_group_id = block_group_at(position)
    if block_group_id is not None:
        hexagons_by_id[position['id']] = {**position, 'geoid': block_group_id, 'assigned_by': 'centre'}

print(f'{count_of(hexagons_by_id):,} grid positions fall inside a block group')

8,568 grid positions fall inside a block group


Two compact urban block groups are small enough to slip between hexagon centers. So neither is left off the map, each is given the hexagon nearest its Census interior point, moved over from a neighboring block group that still keeps at least one hexagon of its own.

In [10]:
def squared_miles_between(first, second):
    return ((second['lon'] - first['lon']) * miles_per_degree_lon) ** 2 + ((second['lat'] - first['lat']) * miles_per_degree_lat) ** 2


hexagon_count_by_block_group = Counter(hexagon['geoid'] for hexagon in hexagons_by_id.values())

for block_group in block_groups:
    if hexagon_count_by_block_group[block_group['geoid']] > 0:
        continue
    nearest_position = min(grid_positions, key=lambda position: squared_miles_between(block_group['interior_point'], position))
    neighbor = hexagons_by_id.get(nearest_position['id'])
    if neighbor is not None:
        assert hexagon_count_by_block_group[neighbor['geoid']] > 1
        hexagon_count_by_block_group[neighbor['geoid']] -= 1
    hexagons_by_id[nearest_position['id']] = {**nearest_position, 'geoid': block_group['geoid'], 'assigned_by': 'interior-point'}
    hexagon_count_by_block_group[block_group['geoid']] = 1
    print(f"block group {block_group['geoid']} gets hexagon {nearest_position['id']}")

block group 391130025004 gets hexagon r56c75
block group 391130011001 gets hexagon r71c58


In [11]:
grid_order = {position['id']: order for order, position in enumerate(grid_positions)}
hexagons = sorted(hexagons_by_id.values(), key=lambda hexagon: grid_order[hexagon['id']])

block_groups_without_a_hexagon = [block_group for block_group in block_groups if hexagon_count_by_block_group[block_group['geoid']] == 0]
every_block_group_has_a_hexagon = count_of(block_groups_without_a_hexagon) == 0

print(f'{count_of(hexagons):,} hexagons')
print(f'block groups with no hexagon: {count_of(block_groups_without_a_hexagon)}')

8,568 hexagons
block groups with no hexagon: 0


Now we check our rebuilt grid against `travel-surface@2026-10-03.json` to confirm it produces the exact same 8,568 hexagons in the same block groups:

In [12]:
travel_surface = json.loads((snapshots_folder / 'travel-surface@2026-10-03.json').read_text())
map_hexagons = travel_surface['cells']

rebuilt_layout = [(hexagon['id'], hexagon['geoid']) for hexagon in hexagons]
map_layout = [(map_hexagon['id'], map_hexagon['geoid']) for map_hexagon in map_hexagons]
same_hexagons = rebuilt_layout == map_layout

print(f'{count_of(map_hexagons):,} hexagons on the map')

8,568 hexagons on the map


**Check:** 8,568 hexagons, the same ones in the same block groups as the map, and every block group has at least one.

In [13]:
assert count_of(hexagons) == 8_568
assert same_hexagons
assert every_block_group_has_a_hexagon

## 3. The households in each hexagon

To estimate how many households live in each hexagon, we combine two Census datasets:

- **2020 Decennial Census blocks** count housing units on every Census block, usually a single city block in town, telling us **where** homes sit.
- **American Community Survey (ACS) block groups** report occupied households and vehicle ownership across groups of a few dozen blocks, telling us **how many** households live there and what share lack a car.

We divide each block group's households across its hexagons in proportion to the 2020 Census housing units inside each hexagon. Because the shares within each block group add up to 100%, every block group keeps its exact household total. Each hexagon also inherits its block group's carless rate, and that part is an estimate: the Census does not say which streets within a block group the carless households live on.

In [14]:
census_blocks = json.loads((snapshots_folder / 'census-blocks@2026-10-06.json').read_text())['blocks']

census_housing_units = sum(housing_units for block_group_id, housing_units, points in census_blocks)

print(f'{count_of(census_blocks):,} lived-in Census blocks, {census_housing_units:,} housing units')

8,384 lived-in Census blocks, 251,523 housing units


In [15]:
hexagons_by_block_group = defaultdict(list)
for hexagon in hexagons:
    hexagons_by_block_group[hexagon['geoid']].append(hexagon)

housing_units_by_hexagon = Counter()
hexagons_reached_by_block = []
for block_group_id, housing_units, points in census_blocks:
    housing_units_per_point = housing_units / count_of(points)
    reached = set()
    for lat, lon in points:
        nearest_hexagon = min(hexagons_by_block_group[block_group_id], key=lambda hexagon: squared_miles_between({'lat': lat, 'lon': lon}, hexagon))
        housing_units_by_hexagon[nearest_hexagon['id']] += housing_units_per_point
        reached.add(nearest_hexagon['id'])
    hexagons_reached_by_block.append({'block_group_id': block_group_id, 'points': count_of(points), 'hexagons_reached': count_of(reached)})

housing_units_by_block_group = Counter()
for hexagon in hexagons:
    housing_units_by_block_group[hexagon['geoid']] += housing_units_by_hexagon[hexagon['id']]

In [16]:
households_by_block_group = {block_group['geoid']: block_group['households'] for block_group in block_groups}


def hexagon_households(hexagon):
    block_group_housing_units = housing_units_by_block_group[hexagon['geoid']]
    if block_group_housing_units == 0:
        return 0
    return households_by_block_group[hexagon['geoid']] * housing_units_by_hexagon[hexagon['id']] / block_group_housing_units


households_by_hexagon = {hexagon['id']: hexagon_households(hexagon) for hexagon in hexagons}

print(f'{sum(households_by_hexagon.values()):,.0f} households shared across the hexagons (county total {county_households:,})')

231,319 households shared across the hexagons (county total 231,319)


In the 84 block groups over 15% carless, where every high-need hexagon on the map is located, Census blocks are compact urban blocks, and **88%** of them put all their homes inside a single hexagon:

In [17]:
high_vehicle_need_block_group_ids = {block_group['geoid'] for block_group in high_vehicle_need_block_groups}
city_blocks = [block for block in hexagons_reached_by_block if block['block_group_id'] in high_vehicle_need_block_group_ids]
city_blocks_in_one_hexagon = count_of([block for block in city_blocks if block['hexagons_reached'] == 1])
city_single_point_blocks = count_of([block for block in city_blocks if block['points'] == 1])

print(f'{count_of(city_blocks):,} lived-in Census blocks in the {count_of(high_vehicle_need_block_group_ids)} block groups over 15% carless')
print(f'all homes in one hexagon: {city_blocks_in_one_hexagon:,} blocks ({city_blocks_in_one_hexagon / count_of(city_blocks):.0%})')
print(f'{city_single_point_blocks:,} of them small enough to be a single point')

1,627 lived-in Census blocks in the 84 block groups over 15% carless
all homes in one hexagon: 1,429 blocks (88%)
1,229 of them small enough to be a single point


**Check:** 1,627 lived-in blocks in those block groups; 1,429 put all their homes in one hexagon, and 1,229 are a single point.

In [18]:
assert count_of(city_blocks) == 1_627
assert city_blocks_in_one_hexagon == 1_429
assert city_single_point_blocks == 1_229

At the other extreme, the county's largest rural block group spans 498 hexagons of mostly farmland, with homes spread thinly across them:

In [19]:
largest_block_group_id, largest_hexagon_count = Counter(hexagon['geoid'] for hexagon in hexagons).most_common(1)[0]
household_shares = sorted((households_by_hexagon[hexagon['id']] for hexagon in hexagons_by_block_group[largest_block_group_id]), reverse=True)

print(f'block group {largest_block_group_id}: {households_by_block_group[largest_block_group_id]:,} households over {largest_hexagon_count} hexagons')
print(f'largest shares: {", ".join(f"{share:.1f}" for share in household_shares[:6])}')
print(f'hexagons with no homes at all: {count_of([share for share in household_shares if share == 0])}')

block group 391130506003: 1,396 households over 498 hexagons
largest shares: 73.9, 50.5, 31.8, 27.2, 20.1, 15.4
hexagons with no homes at all: 1


Next, we verify that our recomputed household counts match `travel-surface@2026-10-03.json` for every hexagon, and that all 422 block groups keep their exact Census totals:

In [20]:
def same_apart_from_rounding(first, second):
    return abs(first - second) < 1e-9


def keeps_household_total(block_group_id):
    shared_out = sum(households_by_hexagon[hexagon['id']] for hexagon in hexagons_by_block_group[block_group_id])
    return housing_units_by_block_group[block_group_id] == 0 or same_apart_from_rounding(shared_out, households_by_block_group[block_group_id])


largest_difference = max(abs(households_by_hexagon[map_hexagon['id']] - map_hexagon['households']) for map_hexagon in map_hexagons)
block_groups_keeping_their_total = count_of([block_group_id for block_group_id in housing_units_by_block_group if keeps_household_total(block_group_id)])

print(f'largest difference from the map: {largest_difference:.9f} households')
print(f'block groups whose hexagons add up to their exact total: {block_groups_keeping_their_total} of {count_of(housing_units_by_block_group)}')

largest difference from the map: 0.000000000 households
block groups whose hexagons add up to their exact total: 422 of 422


**Check:** 8,384 lived-in blocks with 251,523 housing units; every hexagon's households match the map; every block group keeps its exact total; and the county total is unchanged.

In [21]:
assert count_of(census_blocks) == 8_384
assert census_housing_units == 251_523
assert same_apart_from_rounding(largest_difference, 0)
assert block_groups_keeping_their_total == count_of(housing_units_by_block_group)
assert round(sum(households_by_hexagon.values())) == county_households

## 4. The 2,933 lived-in hexagons

To avoid flagging empty farmland or industrial parks, a hexagon counts as **lived-in** only if it has at least **250 households per square mile**: about **14 homes** in a 0.054-square-mile hexagon. Sparse hexagons below that line still show travel times on the map, but are never flagged as high-need.

In [22]:
lived_in_households_per_sq_mi = 250
hexagon_area_sq_mi = math.sqrt(3) / 2 * hexagon_spacing_mi ** 2
lived_in_line_households = lived_in_households_per_sq_mi * hexagon_area_sq_mi

print(f'hexagon area {hexagon_area_sq_mi:.4f} sq mi')
print(f'lived-in line {lived_in_line_households:.2f} households, about {round(lived_in_line_households)} homes')

hexagon area 0.0541 sq mi
lived-in line 13.53 households, about 14 homes


In [23]:
lived_in_hexagons = [
    hexagon for hexagon in hexagons
    if households_by_hexagon[hexagon['id']] > 0 and households_by_hexagon[hexagon['id']] >= lived_in_line_households
]
map_lived_in_ids = {map_hexagon['id'] for map_hexagon in map_hexagons if map_hexagon['populated']}

lived_in_ids = {hexagon['id'] for hexagon in lived_in_hexagons}
lived_in_households = sum(households_by_hexagon[hexagon_id] for hexagon_id in lived_in_ids)

print(f'{count_of(lived_in_hexagons):,} lived-in hexagons, home to {lived_in_households:,.0f} households')
print(f'{count_of(map_lived_in_ids):,} lived-in hexagons on the map')

2,933 lived-in hexagons, home to 216,539 households
2,933 lived-in hexagons on the map


**Check:** the lived-in line is about 14 homes, and the 2,933 lived-in hexagons are exactly the ones the map uses.

In [24]:
assert round(lived_in_line_households) == 14
assert count_of(lived_in_hexagons) == 2_933
assert lived_in_ids == map_lived_in_ids

In the 84 urban block groups over 15% carless, a typical block group spans just **4 lived-in hexagons**. In the 498-hexagon rural block group from section 3, only **9 hexagons** meet the lived-in threshold:

In [25]:
lived_in_hexagons_per_city_block_group = sorted(
    count_of([hexagon for hexagon in hexagons_by_block_group[block_group_id] if hexagon['id'] in lived_in_ids])
    for block_group_id in high_vehicle_need_block_group_ids
)
lived_in_hexagons_in_farmland_group = count_of([hexagon for hexagon in hexagons_by_block_group[largest_block_group_id] if hexagon['id'] in lived_in_ids])

fewest, most = lived_in_hexagons_per_city_block_group[0], lived_in_hexagons_per_city_block_group[-1]
print(f'lived-in hexagons per city block group: median {statistics.median(lived_in_hexagons_per_city_block_group):g}, from {fewest} to {most}')
print(f'block group {largest_block_group_id}: {lived_in_hexagons_in_farmland_group} lived-in hexagons of {largest_hexagon_count}')

lived-in hexagons per city block group: median 4, from 1 to 17
block group 391130506003: 9 lived-in hexagons of 498


**Check:** a median of 4 lived-in hexagons per city block group, and 9 in the farmland block group.

In [26]:
assert statistics.median(lived_in_hexagons_per_city_block_group) == 4
assert lived_in_hexagons_in_farmland_group == 9

## Summary

| Check | Result | Where it appears on the site |
| :--- | :--- | :--- |
| Census block groups | **422**, 231,319 households, 19,669 without a car (8.5%) | Story step 9 ("nearly 9 percent of households") |
| Block groups over 15% carless | **84**, home to 11,137 carless households | Methodology §3, the car-ownership line |
| Hexagons, rebuilt from the boundaries | **8,568**, identical to the map | Methodology §2 |
| Households in each hexagon, rebuilt from 8,384 Census blocks | Identical to the map; every block group keeps its total | Methodology §1 |
| Census blocks whose homes all land in one hexagon, in the 84 block groups over 15% carless | **1,429 of 1,627** (88%); a median of 4 lived-in hexagons per block group | Methodology §4 |
| Lived-in hexagons (250 households per sq mi, about 14 homes) | **2,933**, identical to the map | Story step 24, Methodology §2 |

The cell below outputs these key figures as JSON, so they can be checked against the numbers the live site shows.

In [27]:
audited_figures = {
    'blockGroups': count_of(block_groups),
    'households': county_households,
    'carlessHouseholds': county_carless_households,
    'highVehicleNeedBlockGroups': count_of(high_vehicle_need_block_groups),
    'highVehicleNeedCarlessHouseholds': high_vehicle_need_carless_households,
    'hexagons': count_of(hexagons),
    'livedInHexagons': count_of(lived_in_hexagons),
}

print(json.dumps(audited_figures, sort_keys=True))

{"blockGroups": 422, "carlessHouseholds": 19669, "hexagons": 8568, "highVehicleNeedBlockGroups": 84, "highVehicleNeedCarlessHouseholds": 11137, "households": 231319, "livedInHexagons": 2933}


To rerun every audit notebook without Jupyter: `python3 notebooks/run_notebooks.py` from the project folder. It needs only Python 3.10 or newer.